# M7 Run 5 - A/B regularization

Pushes further on the overfit gap now that strong augmentation helped. **Fixed**: imbalance `both` + augmentation `strong`. Varies **dropout x weight-decay**: base(0.3,1e-4) / drop50(0.5,1e-4) / wd1e3(0.3,1e-3) / both_reg(0.5,1e-3).

Same mini-OOF proxy (train 5-8, test 1-4 ~58 WPW, 3 seeds @5000, reuse Run 2 cache, checkpointed).

> Decision rule = the override criterion declared at Run 4: adopt a non-baseline variant only if per-seed mean beats baseline by ~3 sigma AND per-seed std does not increase; else keep base. Winner confirmed in full OOF (Run 7).

### Block 5.0: Setup and A/B grid

In [ ]:
# M7 Run 5 - A/B REGULARIZATION (push further on the overfit gap, now that strong aug helped).
# Fixed: imbalance 'both' + augmentation 'strong'. Vary dropout x weight-decay (regularization strength).
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings('ignore')
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
METRICS=os.path.join(ROOT,'reports','metrics'); MODELS=os.path.join(ROOT,'models','M7_run5')
CACHE_DIR=os.path.join(PROCESSED,'m7_run2_cache')
for d in (METRICS,MODELS): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC)
from evaluation import _boot_ci
RANDOM_STATE=42; RESO=5000
SEEDS=[0,1,2]
EPOCHS=60; PATIENCE=10; BATCH=64; LR=1e-3
FOCAL_GAMMA=2.0; FOCAL_ALPHA=0.5; VAL_FRAC=0.15; N_NEG_TRAIN=12000; N_JOBS=6
TEST_FOLDS=[1,2,3,4]; TRAIN_FOLDS=[5,6,7,8]; AUG='strong'
# (name, dropout, weight_decay). 'base' = current (drop 0.3, wd 1e-4) under strong aug.
REG_VARIANTS=[('base',0.3,1e-4),('drop50',0.5,1e-4),('wd1e3',0.3,1e-3),('both_reg',0.5,1e-3)]
BASELINE='base'
print('M7 Run 5 | A/B regularization | fixed both+strong | mini-OOF train %s -> test %s @ %d'%(TRAIN_FOLDS,TEST_FOLDS,RESO))
print('variants:', [v[0] for v in REG_VARIANTS]); print('Block 5.0 OK.')


### Block 5.1: Reuse Run 2 cache + mini-OOF split

In [ ]:
# Reuse Run 2 cache; mini-OOF split. X18 is memory-mapped (avoids a 6 GB contiguous allocation).
import shutil, zipfile
NPZ=os.path.join(CACHE_DIR,'m7_run2_data.npz')
def _mmap_member(name):
    # extract an uncompressed npz member to a standalone .npy (streamed, no full-RAM load), then mmap it
    out=os.path.join(CACHE_DIR,name+'.npy')
    if not os.path.exists(out):
        with zipfile.ZipFile(NPZ) as zf, zf.open(name+'.npy') as src, open(out,'wb') as dst:
            shutil.copyfileobj(src,dst,length=16*1024*1024)
    return np.load(out,mmap_mode='r')
X18=_mmap_member('X18')                                   # memory-mapped, NOT loaded into RAM
z=np.load(NPZ,allow_pickle=True); fold18=z['fold18']; y18=z['y18']; z.close()
tr_all=np.where(np.isin(fold18,TRAIN_FOLDS))[0]; te_idx=np.where(np.isin(fold18,TEST_FOLDS))[0]
yte=y18[te_idx]; Xte=np.ascontiguousarray(X18[te_idx])    # copies only folds 1-4 (~3.2 GB)
print('mini-OOF | train pool %d (%d WPW) | test %d (%d WPW) | X18 mmap %s'%(
    len(tr_all),int(y18[tr_all].sum()),len(te_idx),int(yte.sum()),X18.shape))
print('Block 5.1 OK.')

### Block 5.2: 1D-ResNet (identical)

In [ ]:
import torch, torch.nn as nn
torch.set_num_threads(N_JOBS)
class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)
class ResNet1d(nn.Module):
    def __init__(self,ch=(16,32,64),pdrop=0.3):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1); self.layer2=BasicBlock1d(ch[0],ch[1],2); self.layer3=BasicBlock1d(ch[1],ch[2],2)
        self.pool=nn.AdaptiveMaxPool1d(1); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(pdrop),nn.Linear(ch[2],1))
    def forward(self,x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x)))))).squeeze(1)
print('Block 5.2 OK | params:', sum(p.numel() for p in ResNet1d().parameters()))


### Block 5.3: Fixed strong aug + training with dropout/WD

In [ ]:
# Fixed strong augmentation. train_one takes dropout + weight_decay (the A/B axes).
def focal_loss(logits,targets,gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA):
    ce=nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=torch.where(targets==1,p,1-p)
    a=torch.where(targets==1,torch.full_like(targets,alpha),torch.full_like(targets,1-alpha))
    return (a*(1-pt).pow(gamma)*ce).mean()
def augment(xb):   # 'strong'
    T=xb.shape[2]; P=dict(shift=0.08,scale=0.3,noise=0.03,wander=0.08,ldrop=0.45)
    sh=max(1,int(P['shift']*T)); xb=torch.roll(xb,int(torch.randint(-sh,sh+1,(1,)).item()),dims=2)
    xb=xb*((1.0-P['scale'])+2*P['scale']*torch.rand(xb.shape[0],1,1)); xb=xb+P['noise']*torch.randn_like(xb)
    t=torch.linspace(0,1,T).view(1,1,-1); fb=0.5+2.0*torch.rand(xb.shape[0],1,1); ph=2*np.pi*torch.rand(xb.shape[0],1,1)
    xb=xb+P['wander']*torch.sin(2*np.pi*fb*t+ph)
    if torch.rand(1).item()<P['ldrop']: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    if torch.rand(1).item()<0.25: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    return xb
def predict(model,X):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),256):
            out.append(torch.sigmoid(model(torch.tensor(np.ascontiguousarray(X[i:i+256],dtype=np.float32)))).numpy())
    return np.nan_to_num(np.concatenate(out))
def train_one(seed,Xtr,ytr,Xva,yva,pdrop,wd):
    torch.manual_seed(seed); np.random.seed(seed); rng=np.random.default_rng(seed)
    pos=np.where(ytr==1)[0]; neg=np.where(ytr==0)[0]; half=BATCH//2
    Xv=torch.tensor(np.ascontiguousarray(Xva,dtype=np.float32)); yv=torch.tensor(yva)
    model=ResNet1d(pdrop=pdrop); opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=wd)
    steps=max(1,len(ytr)//BATCH); best=1e9; best_state=None; bad=0
    for ep in range(EPOCHS):
        model.train()
        for _ in range(steps):
            bi=np.concatenate([rng.choice(pos,half,True),rng.choice(neg,half,True)])
            xb=augment(torch.tensor(np.ascontiguousarray(Xtr[bi],dtype=np.float32))); yb=torch.tensor(ytr[bi])
            opt.zero_grad(); loss=focal_loss(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=[focal_loss(model(Xv[i:i+256]),yv[i:i+256]).item() for i in range(0,len(Xv),256)]
        vloss=float(np.mean(vl))
        if vloss<best-1e-4: best=vloss; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state); return model
print('Block 5.3 OK.')


### Block 5.4: A/B loop (checkpointed per variant+seed)

In [ ]:
def build_pool(seed0):
    rng=np.random.default_rng(seed0)
    pos=tr_all[y18[tr_all]==1]; negall=tr_all[y18[tr_all]==0]
    neg=rng.choice(negall,min(N_NEG_TRAIN,len(negall)),replace=False)
    vp=pos.copy(); vn=neg.copy(); rng.shuffle(vp); rng.shuffle(vn)
    nvp=max(6,int(VAL_FRAC*len(vp))); nvn=int(VAL_FRAC*len(vn))
    val=np.concatenate([vp[:nvp],vn[:nvn]]); pool=np.concatenate([pos,neg]); tr=np.setdiff1d(pool,val)
    return tr,val
tr_idx,val_idx=build_pool(2024)
Xtr=np.ascontiguousarray(X18[tr_idx],dtype=np.float32); ytr=y18[tr_idx]; Xva=X18[val_idx]; yva=y18[val_idx]
print('pool: train %d (%d WPW) | val %d (%d WPW)'%(len(tr_idx),int(ytr.sum()),len(val_idx),int(yva.sum())))
CK=os.path.join(MODELS,'run5_ab_ckpt.npz'); store={}
if os.path.exists(CK):
    store=np.load(CK,allow_pickle=True)['store'].item(); print('[ckpt] resumed:',list(store.keys()))
rows=[]; t0=time.time()
for name,pdrop,wd in REG_VARIANTS:
    seed_pred=[np.asarray(p) for p in store.get(name,{}).get('pred',[])]; done=store.get(name,{}).get('done',0)
    for si,s in enumerate(SEEDS):
        if si<done: continue
        model=train_one(s,Xtr,ytr,Xva,yva,pdrop,wd)
        seed_pred.append(predict(model,Xte))
        store[name]={'pred':[p.tolist() for p in seed_pred],'done':si+1}; np.savez(CK,store=np.array(store,dtype=object))
    ens=np.mean(seed_pred,0); ap=float(average_precision_score(yte,ens)); auc=float(roc_auc_score(yte,ens))
    lo,hi=_boot_ci(yte.astype(int),ens,average_precision_score); seed_ap=[float(average_precision_score(yte,p)) for p in seed_pred]
    rows.append(dict(variant=name,dropout=pdrop,wd=wd,AP=ap,AP_lo=lo,AP_hi=hi,AUC=auc,
                     AP_seed_mean=float(np.mean(seed_ap)),AP_seed_std=float(np.std(seed_ap))))
    print('  [%-8s] drop %.1f wd %.0e | AP %.3f CI[%.3f,%.3f] | per-seed %.3f+/-%.3f | %.1f min'%(
        name,pdrop,wd,ap,lo,hi,np.mean(seed_ap),np.std(seed_ap),(time.time()-t0)/60))
ab=pd.DataFrame(rows)
print('Block 5.4 OK.'); print(ab.to_string(index=False))


### Block 5.5: Ranking + decision (3-sigma override criterion)

In [ ]:
# Override criterion (declared at Run 4): adopt a non-baseline variant only if per-seed mean beats baseline
# by ~3 sigma AND per-seed std does not increase; else keep baseline. (mini-OOF bootstrap CI is too wide at 58 WPW.)
base=ab[ab.variant==BASELINE].iloc[0]; ab_sorted=ab.sort_values('AP_seed_mean',ascending=False).reset_index(drop=True); top=ab_sorted.iloc[0]
def se(std): return std/np.sqrt(len(SEEDS))
sigma=np.sqrt(se(base.AP_seed_std)**2+se(top.AP_seed_std)**2)
delta=top.AP_seed_mean-base.AP_seed_mean
strong_win=bool(top.variant!=BASELINE and delta>3*sigma and top.AP_seed_std<=base.AP_seed_std)
chosen=top.variant if strong_win else BASELINE
print('=== M7 Run 5 - regularization A/B (mini-OOF, test folds 1-4) ===')
print(ab.sort_values('AP',ascending=False).to_string(index=False))
print('  baseline %s: AP %.3f | per-seed %.3f+/-%.3f'%(BASELINE,base.AP,base.AP_seed_mean,base.AP_seed_std))
print('  top per-seed: %s %.3f (delta %+.3f vs base, ~%.1f sigma) | std %.3f vs base %.3f'%(
    top.variant,top.AP_seed_mean,delta,delta/max(sigma,1e-9),top.AP_seed_std,base.AP_seed_std))
print('  beats by >=3 sigma AND not less stable ? %s -> ADOPT: %s'%(strong_win,chosen))
json.dump({'variants':ab.sort_values('AP',ascending=False).to_dict('records'),'baseline':BASELINE,
           'winner':top.variant,'delta_sigma':float(delta/max(sigma,1e-9)),'adopted':chosen},
          open(os.path.join(METRICS,'m7_run5_regularization_ab.json'),'w'),indent=2,default=float)
print('Block 5.5 OK | adopted regularization: %s'%chosen)
